# 🕵️‍♂️ Analisis Silang Tebakan Test Set (LoRA Ensemble vs Last Layer vs kNN)
Notebook ini sifatnya *stand-alone*. Kita akan memuat model LoRA 5-Fold dan Last Layer Fold 0, menebak seluruh gambar *Test Set*, lalu membandingkan kombinasinya.

In [ ]:
import os, sys, shutil, glob
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Clone atau Pull Repo Terbaru
REPO_DIR = '/content/satria-data-bdcugm02'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# 3. Upgrade Torchao (WAJIB untuk memuat arsitektur LoRA)
!pip install -q -U "torchao>=0.16.0"
!pip install -q --no-warn-conflicts -r {REPO_DIR}/track_b/requirements.txt

In [ ]:
# 4. Copy Cepat Gambar TEST ke Storage Lokal Colab (/tmp) agar I/O Super Cepat
import os
if os.path.exists('/content/drive/MyDrive/BDC2026/test'):
    DRIVE_TEST_DIR = '/content/drive/MyDrive/BDC2026/test'
else:
    DRIVE_TEST_DIR = '/content/drive/MyDrive/BDC2026apace/test'
LOCAL_TEST_DIR = '/tmp/dataset/test'

def copy_img_worker(args):
    src, dst = args
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
        shutil.copy2(src, dst)

if os.path.exists(DRIVE_TEST_DIR):
    print(f"🚀 Memulai copy cepat gambar TEST dari {DRIVE_TEST_DIR} ke storage lokal Colab (/tmp)...")
    test_imgs = [f for f in glob.glob(os.path.join(DRIVE_TEST_DIR, "*")) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    files_to_copy = [(src, os.path.join(LOCAL_TEST_DIR, os.path.basename(src))) for src in test_imgs]
    
    with ThreadPoolExecutor(max_workers=32) as executor:
        list(executor.map(copy_img_worker, files_to_copy))
    print(f"✅ Selesai meng-copy {len(files_to_copy)} gambar Test ke {LOCAL_TEST_DIR}")
else:
    print(f"⚠️ Folder {DRIVE_TEST_DIR} tidak ditemukan!")


In [ ]:
# 5. Setup Path & Load kNN Submission
sys.path.insert(0, os.path.join(REPO_DIR, 'track_a', 'src'))
sys.path.insert(0, os.path.join(REPO_DIR, 'track_b', 'src'))
sys.path.insert(0, os.path.join(REPO_DIR, 'track_b', 'experiments'))

import torch
import pandas as pd
import numpy as np
import config, lora_ft
from config import CFG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

KNN_SUBMISSION_PATH = os.path.join(CFG.save_dir, "..", "output_trackC", "submission_apace.csv")
if not os.path.exists(KNN_SUBMISSION_PATH):
    print(f"⚠️ {KNN_SUBMISSION_PATH} tidak ditemukan, pastikan path benar!")
else:
    knn_df = pd.read_csv(KNN_SUBMISSION_PATH)
    print(f"Loaded kNN submission: {len(knn_df)} baris")


In [ ]:
# 6. Siapkan Test Loader dari Lokal (/tmp)
from transformers import AutoImageProcessor, AutoModel
from dataset import WasteDataset
from transforms import build_transforms
from torch.utils.data import DataLoader

CHECKPOINT = 'google/siglip2-so400m-patch14-384'
processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
data_config = lora_ft.hf_processor_to_data_config(processor)

local_files = glob.glob(os.path.join(LOCAL_TEST_DIR, "*"))
name_to_path = {os.path.splitext(os.path.basename(f))[0]: f for f in local_files}

test_images = []
for img_id in knn_df['id']:
    base_id = str(img_id)
    if base_id.endswith(('.jpg', '.png', '.jpeg')):
        base_id = os.path.splitext(base_id)[0]
    
    if base_id in name_to_path:
        test_images.append(name_to_path[base_id])
    else:
        test_images.append(os.path.join(LOCAL_TEST_DIR, f"{base_id}.jpg"))

test_df = pd.DataFrame({'filepath': test_images, 'label': 0})

eval_tfm = build_transforms(data_config, 384, train=False)
test_ds = WasteDataset(test_df, transform=eval_tfm)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)


In [ ]:
# 7. Fungsi Pembantu Memuat Model
from lora_ft import build_variant
import gc

def load_model_weights(variant, ckpt_path):
    print(f"\n📦 Memuat model {variant} dari {os.path.basename(ckpt_path)}...")
    encoder = AutoModel.from_pretrained(CHECKPOINT).vision_model
    hidden_size = encoder.config.hidden_size
    n_last_blocks = 4 if variant == 'lora' else None
    model, _ = build_variant(variant, encoder, hidden_size, num_classes=3, n_last_blocks=n_last_blocks)
    
    state_dict = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state_dict, strict=False)
    model = model.to(device)
    model.eval()
    return model


In [ ]:
# 8. Inferensi Ensemble 5-Fold LoRA (Aman untuk RAM)
def predict_lora_5fold():
    lora_files = [f"lora_ft_fold{i}_5ep_v3_best.pt" for i in range(5)]
    for lf in lora_files:
        if not os.path.exists(os.path.join(CFG.save_dir, lf)):
            print(f"❌ {lf} tidak ditemukan di Drive!")
            return None

    total_samples = len(test_loader.dataset)
    ensemble_probs = torch.zeros((total_samples, 3), device='cpu')
    
    print("\n🔥 Mulai Ensemble 5-Fold LoRA...")
    for lf in lora_files:
        ckpt_path = os.path.join(CFG.save_dir, lf)
        model = load_model_weights('lora', ckpt_path)
        
        print(f"Menebak Test Set dengan {lf}...")
        ptr = 0
        with torch.no_grad():
            for images, _ in test_loader:
                images = images.to(device)
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(images)
                probs = torch.softmax(logits.float(), dim=-1).cpu()
                
                batch_len = len(images)
                ensemble_probs[ptr : ptr+batch_len] += probs
                ptr += batch_len
                
        del model
        gc.collect()
        torch.cuda.empty_cache()
        
    avg_probs = ensemble_probs / 5.0
    return avg_probs.argmax(dim=1).numpy()

preds_lora_5fold = predict_lora_5fold()
if preds_lora_5fold is not None:
    knn_df['pred_lora5'] = preds_lora_5fold


In [ ]:
# 9. Inferensi Last Layer Fold 0
def predict_last_layer():
    ckpt_path = os.path.join(CFG.save_dir, "last_layer_ft_fold0_v3_best.pt")
    if not os.path.exists(ckpt_path):
        print(f"\n❌ {ckpt_path} tidak ditemukan!")
        return None
        
    model = load_model_weights('last_layer', ckpt_path)
    
    print("Menebak Test Set dengan Last Layer Fold 0...")
    all_preds = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return np.concatenate(all_preds)

preds_last_layer = predict_last_layer()
if preds_last_layer is not None:
    knn_df['pred_ll'] = preds_last_layer


In [ ]:
# 10. Perbandingan Lengkap 4 Skenario
if 'pred_lora5' in knn_df.columns and 'pred_ll' in knn_df.columns:
    print("\n" + "="*50)
    print("📊 HASIL PERBANDINGAN SILANG MODEL")
    print("==================================================")
    
    # 1. LoRA vs Last Layer
    diff_1 = knn_df[knn_df['pred_lora5'] != knn_df['pred_ll']]
    print(f"\n1. LoRA 5-Fold vs Last Layer Fold 0")
    print(f"   🔥 Beda Tebakan: {len(diff_1)} gambar ({len(diff_1)/len(knn_df)*100:.1f}%)")
    
    # 2. LoRA vs kNN
    diff_2 = knn_df[knn_df['pred_lora5'] != knn_df['predicted']]
    print(f"\n2. LoRA 5-Fold vs kNN")
    print(f"   🔥 Beda Tebakan: {len(diff_2)} gambar ({len(diff_2)/len(knn_df)*100:.1f}%)")
    
    # 3. Last Layer vs kNN
    diff_3 = knn_df[knn_df['pred_ll'] != knn_df['predicted']]
    print(f"\n3. Last Layer Fold 0 vs kNN")
    print(f"   🔥 Beda Tebakan: {len(diff_3)} gambar ({len(diff_3)/len(knn_df)*100:.1f}%)")
    
    # 4. 3-Way Analysis
    print(f"\n4. Analisis 3-Way (LoRA vs LL vs kNN)")
    sepakat_semua = knn_df[(knn_df['pred_lora5'] == knn_df['predicted']) & (knn_df['pred_ll'] == knn_df['predicted'])]
    print(f"   🤝 Ketiganya Sepakat: {len(sepakat_semua)} gambar ({len(sepakat_semua)/len(knn_df)*100:.1f}%)")
    
    lora_ll_vs_knn = knn_df[(knn_df['pred_lora5'] == knn_df['pred_ll']) & (knn_df['pred_lora5'] != knn_df['predicted'])]
    print(f"   ⚔️ LoRA & LL Sepakat melawan kNN: {len(lora_ll_vs_knn)} gambar")
    
    beda_semua = knn_df[(knn_df['pred_lora5'] != knn_df['predicted']) & 
                        (knn_df['pred_ll'] != knn_df['predicted']) & 
                        (knn_df['pred_lora5'] != knn_df['pred_ll'])]
    print(f"   💥 Ketiganya Saling Berbeda (0, 1, 2): {len(beda_semua)} gambar")
    
    if len(beda_semua) > 0:
        print("\nDaftar gambar yang ketiganya menebak berbeda:")
        display(beda_semua[['id', 'predicted', 'pred_lora5', 'pred_ll']].rename(
            columns={'predicted': 'kNN', 'pred_lora5': 'LoRA_5F', 'pred_ll': 'LastLayer_F0'}
        ))
